# ML Pipeline 2: Post Engagement + Donation Referrals
Predictive + explanatory modeling for social post performance.

In [ ]:
import os, sys
from pathlib import Path
import pandas as pd
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
from src.db import load_env, build_engine
from src.features import build_post_frame, feature_target_frames
from src.modeling import train_engagement_models, train_referral_models


## Business KPI Mapping

- Stakeholder owner: Outreach and Communications Lead
- Decision enabled: schedule platform/time/content combinations with highest expected response
- Primary KPIs: average engagement rate, donation referrals per 100 posts
- Guardrail KPIs: boost spend efficiency, negative sentiment incidents
- Minimum success criteria: +10% referral rate lift at same or lower cost-per-referral

## Problem Framing
- Predictive: estimate engagement and referral likelihood.
- Explanatory: identify post characteristics associated with stronger outcomes.

In [ ]:
load_env('.env')
engine = build_engine(os.getenv('DB_PROFILE', 'local'))
df = build_post_frame(engine)
X, y_eng, y_ref_raw, y_ref_bin = feature_target_frames(df)
print(df.shape)
df.head()

In [ ]:
# Data Understanding Audit: missingness + anomaly checks
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
display(missing_pct.head(10).to_frame('missing_pct'))

audit = {
    'rows': len(df),
    'negative_engagement_rows': int((df['engagement_rate'] < 0).sum()),
    'referrals_gt_impressions_rows': int(((df['donation_referrals'] > 0) & (df['impressions'] == 0)).sum()) if 'impressions' in df.columns else 0,
    'post_hour_out_of_range_rows': int(((df['post_hour'] < 0) | (df['post_hour'] > 23)).sum()),
}
print('Audit summary:', audit)

print('Feature rationale: platform/time/content/cta/boost variables map directly to controllable campaign decisions.')

In [ ]:
import numpy as np
import subprocess
import sys

try:
    import seaborn as sns
    import matplotlib.pyplot as plt
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'seaborn', 'matplotlib'])
    import seaborn as sns
    import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x='platform', y='engagement_rate', ax=axes[0])
axes[0].set_title('Engagement by Platform')
axes[0].tick_params(axis='x', rotation=45)

hourly = df.groupby('post_hour', as_index=False)['donation_referrals'].mean()
sns.lineplot(data=hourly, x='post_hour', y='donation_referrals', marker='o', ax=axes[1])
axes[1].set_title('Avg Donation Referrals by Posting Hour')
plt.tight_layout()
plt.show()

In [ ]:
eng_results, eng_best_model = train_engagement_models(X, y_eng)
print('Predictive engagement results:')
display(eng_results)

In [ ]:
ref_metrics, ref_model, ref_coef = train_referral_models(X, y_ref_bin)
print('Predictive referral metrics:', {k: round(v,4) for k,v in ref_metrics.items()})
print('Explanatory drivers of referral probability (top positive):')
display(ref_coef.head(12))
print('Top negative:')
display(ref_coef.tail(12).sort_values('coefficient'))

In [ ]:
# Threshold tuning + FP/FN cost table for referral classifier
ref_proba_full = ref_model.predict_proba(X)[:, 1]
thresholds = np.arange(0.1, 0.95, 0.05)
fp_cost = 1.0
fn_cost = 3.0
rows = []
for t in thresholds:
    pred = (ref_proba_full >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_ref_bin, pred, labels=[0, 1]).ravel()
    total_cost = fp_cost * fp + fn_cost * fn
    rows.append({'threshold': round(float(t), 2), 'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn), 'total_cost': float(total_cost)})

cost_df = pd.DataFrame(rows).sort_values('total_cost').reset_index(drop=True)
display(cost_df.head(10))
best_t = float(cost_df.loc[0, 'threshold'])
print(f'Selected threshold by cost minimization: {best_t:.2f}')

## Evaluation In Business Terms
- False positives: extra outreach spend on low-response posts.
- False negatives: missed opportunities for high-value campaigns.
- Use this model to prioritize posting schedules/platform-content combos.

## Operationalization Policy + Monitoring

- Threshold policy: select threshold minimizing FP/FN cost table from this notebook; default fallback 0.45.
- Action bands: high-priority outreach list for top probability decile; medium for A/B testing; low for holdout/control.
- Retraining cadence: monthly retrain or earlier if referral PR-AUC drops >15%.
- Monitoring references:
  - `ml-pipelines/integration/pipeline_registry.yaml`
  - `ml-pipelines/integration/monitoring_spec.md`
  - `ml-pipelines/integration/README.md`

## Causal Caveat
These coefficients are associations from observational data; they do not prove causality.

## Deployment Notes
Expose predictions in the app as a campaign planning widget: input post attributes, output expected engagement and referral likelihood.